# 📝 Prompt Engineering for RAG

## Why Prompt Design Matters

The LLM is only as good as what you put in the prompt.  
With the same retrieved documents, a better prompt gives a better answer.

In this notebook we go from a naive prompt to a production-ready one, step by step.

In [ ]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()  # reads ANTHROPIC_API_KEY from .env if present

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def llm(prompt, max_tokens=400):
    """Call Claude Haiku. Fast and cheap — perfect for RAG pipelines."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

print("✅ Anthropic client ready!")
print("   Model: claude-haiku-4-5-20251001")
print("   Set ANTHROPIC_API_KEY in .env or environment.")


In [ ]:
# Use the llm() helper defined above to call Claude
# It's already set up — just call llm(prompt) anywhere in this notebook
print("✅ Using real Claude calls via llm() helper")


## Our Sample Data

We'll use these retrieved docs throughout the notebook.

In [ ]:
# Retrieved documents (output from the retrieval step)
retrieved_docs = [
    "RAG stands for Retrieval-Augmented Generation. It combines a retriever with an LLM.",
    "The retriever finds relevant documents from a knowledge base using embedding similarity.",
    "The retrieved documents are passed as context in the LLM prompt to ground the response.",
    "RAG reduces hallucination because the LLM is anchored to real retrieved facts.",
]

question = "What is RAG and why does it help reduce hallucination?"

print(f"Question: {question}\n")
print("Retrieved docs:")
for i, d in enumerate(retrieved_docs, 1):
    print(f"  {i}. {d}")

## 1. Level 1 — Naive Prompt (Don't Use This)

In [ ]:
# ❌ Naive: just dumps docs in, no structure

naive_prompt = retrieved_docs[0] + " " + retrieved_docs[1] + " " + question

print("Naive prompt:")
print(naive_prompt)
print("\nProblems:")
print("  - No instruction on how to use the context")
print("  - LLM doesn't know where context ends and question begins")
print("  - No fallback if context doesn't have the answer")

## 2. Level 2 — Basic Structured Prompt

In [ ]:
# ✅ Better: clear structure with labeled sections

context = "\n".join(retrieved_docs)

basic_prompt = f"""Use the following context to answer the question.
Only use information from the context. If the answer is not there, say "I don't know".

Context:
{context}

Question: {question}
Answer:"""

print(basic_prompt)

## 3. Level 3 — Numbered Sources

In [ ]:
# ✅ Even better: number the sources so the LLM can reference them

numbered_context = ""
for i, doc in enumerate(retrieved_docs, 1):
    numbered_context += f"[{i}] {doc}\n"

numbered_prompt = f"""Answer the question using only the numbered sources below.
When you use information from a source, cite it like this: [1], [2], etc.
If the answer cannot be found in the sources, say "Not covered in the provided context."

Sources:
{numbered_context}
Question: {question}
Answer:"""

print(numbered_prompt)

In [ ]:
# What a real LLM would return with this prompt:
example_cited_answer = (
    "RAG (Retrieval-Augmented Generation) combines a retriever with an LLM [1]. "
    "The retriever finds relevant documents using embedding similarity [2], "
    "which are then passed as context in the LLM prompt [3]. "
    "This reduces hallucination because the LLM is anchored to real retrieved facts [4]."
)

print("Example answer with citations:")
print(example_cited_answer)

## 4. Level 4 — System + User Split (Best Practice)

In [ ]:
# ✅ Production pattern: split into system prompt + user message
# System prompt sets the behaviour. User message has the context + question.

system_prompt = """You are a helpful assistant that answers questions based on provided context.
Rules:
- Only use information from the context provided.
- Cite sources using [1], [2] notation.
- If the answer is not in the context, say: "I don't have enough information to answer this."
- Be concise and factual. Do not add information beyond what the context says."""

user_message = f"""Context:
{numbered_context}
Question: {question}"""

print("System prompt:")
print("-" * 50)
print(system_prompt)

print("\nUser message:")
print("-" * 50)
print(user_message)

In [ ]:
# How to call the LLM with system + user split

# from anthropic import Anthropic
# client = Anthropic()
#
# response = client.messages.create(
#     model="claude-sonnet-4-6",
#     max_tokens=500,
#     system=system_prompt,                          # ← system goes here
#     messages=[{"role": "user", "content": user_message}]  # ← user message here
# )
#
# answer = response.content[0].text

print("Code structure ready — uncomment and run with your API key.")

## 5. Prompt Template Comparison

In [ ]:
comparison = """
┌────────────────────┬──────────────────────────────────────────────┐
│ Level              │ What It Does                                 │
├────────────────────┼──────────────────────────────────────────────┤
│ 1. Naive           │ Dumps everything in one string — avoid this  │
│ 2. Basic           │ Clear sections, fallback instruction         │
│ 3. Numbered sources│ LLM can cite specific sources [1][2][3]      │
│ 4. System/User     │ Separates behaviour rules from content       │
│                    │ Best for production, most LLM APIs support   │
└────────────────────┴──────────────────────────────────────────────┘

Recommendation: Start with Level 2, move to Level 4 in production.
"""
print(comparison)

## Key Takeaways 🎯

- Always tell the LLM **where the context is** and **to use it**
- Always include a **fallback instruction** ("say I don't know if not in context")
- **Number your sources** so the LLM can cite them
- Use **system + user split** for production — cleaner separation of concerns
- **Order matters** — put the most relevant doc first

---
Next: `02_Context_Window_Management.ipynb` — what to do when your context is too long.